[![Homepage](https://img.shields.io/badge/homepage-blueviolet?logo=htmx)](https://www.bendai.org/CUHK-STAT3009/)
[![GitHub](https://img.shields.io/badge/GitHub-black.svg?logo=github)](https://github.com/statmlben/CUHK-STAT3009)
[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/statmlben/CUHK-STAT3009/blob/main/nb/warmup/NB_Warmup_06_NN_Recap.ipynb)

# NB Warm-up 06 — Building networks in PyTorch: bricks → `Sequential` → `Module`

**STAT3009 · Recommender Systems**  
**Suggested time:** 15–20 minutes

> **Recall warm-up:** a network is a stack of layers. Each layer has an input shape and an output shape. We will rebuild that idea with two small examples before today's optimization section.

**Route:** the three bricks (4 min) → bottleneck example (4 min) → residual example (7 min) → exit ticket (3 min). No dataset download is needed.

## 1. The three bricks

| brick | what it does | shape |
|---|---|---|
| `nn.Linear(d_in, d_out)` | affine map $xW^\top+b$ | `(B,d_in) → (B,d_out)` |
| `nn.ReLU()` | elementwise $\max(0,z)$ | unchanged |
| `nn.Sequential(...)` | calls layers in order | determined by the layers |
| `nn.Module` | custom `__init__` + `forward` | custom path |

A `Linear(d_in, d_out)` brick has $d_{\rm in}d_{\rm out}+d_{\rm out}$ learned numbers.

In [ ]:
import torch
from torch import nn

torch.manual_seed(0)

X = torch.randn(32, 8)  # 32 examples, 8 features
layer = nn.Linear(8, 4)

print(layer)
print('weight shape:', tuple(layer.weight.shape))
print('bias shape:  ', tuple(layer.bias.shape))
print('output shape:', tuple(layer(X).shape))
print('learned numbers:', sum(p.numel() for p in layer.parameters()))

manual = X @ layer.weight.T + layer.bias
print('same as the formula:', torch.allclose(layer(X), manual))

## 2. Example 1: a bottleneck stack

Build the stack explicitly:

```text
Linear(8,16) → ReLU → Linear(16,4) → ReLU → Linear(4,1)
```

The middle layer is narrow: it forces the model to compress the 8 input features into 4 numbers before producing the prediction.

| brick | output shape | learned numbers |
|---|---|---|
| `Linear(8,16)` | `(B,16)` | $8\cdot16+16=144$ |
| `ReLU()` | `(B,16)` | 0 |
| `Linear(16,4)` | `(B,4)` | $16\cdot4+4=68$ |
| `ReLU()` | `(B,4)` | 0 |
| `Linear(4,1)` | `(B,1)` | $4\cdot1+1=5$ |

Total: **217** learned numbers.

In [ ]:
class BottleneckMLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(8, 16), nn.ReLU(),
            nn.Linear(16, 4), nn.ReLU(),
            nn.Linear(4, 1),
        )

    def forward(self, x):
        return self.net(x)


bottleneck = BottleneckMLP()
print(bottleneck)
print('total learned numbers:',
      sum(p.numel() for p in bottleneck.parameters()))
print('output shape:', tuple(bottleneck(X).shape))

### Checkpoint

1. Why is the output shape `(32,1)` rather than `(32,4)`?
2. How many learned numbers are in the whole model?
3. What is the difference between `nn.Sequential` and a plain Python list of layers?

## 3. Example 2: a residual connection

A residual block adds the input back to the block output:

$$
h=\text{block}(x)+x,
\qquad
\widehat y=\text{head}(h).
$$

For the addition to work, `block(x)` and `x` must have the **same shape**. Here both are `(B,8)`.

```text
x ──→ Linear(8,8) → ReLU → Linear(8,8) → ReLU ──┐
└──────────────────── add ──────────────────────┴─→ Linear(8,1) → ŷ
```

### Your turn

Complete `forward`: compute the block output, add the original input, then pass the result to the head. You can also complete the parameter-count print.

In [ ]:
class ResidualMLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.block = nn.Sequential(
            nn.Linear(8, 8), nn.ReLU(),
            nn.Linear(8, 8), nn.ReLU(),
        )
        self.head = nn.Linear(8, 1)

    def forward(self, x):
        # h = ...
        # return ...
        ...


# After completing the class, uncomment and run:
# residual = ResidualMLP()
# print(residual)
# print('total learned numbers:',
#       sum(p.numel() for p in residual.parameters()))
# print('block output:', tuple(residual.block(X).shape))
# print('final output:', tuple(residual(X).shape))

### Solution

In [ ]:
class ResidualMLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.block = nn.Sequential(
            nn.Linear(8, 8), nn.ReLU(),
            nn.Linear(8, 8), nn.ReLU(),
        )
        self.head = nn.Linear(8, 1)

    def forward(self, x):
        h = self.block(x) + x
        return self.head(h)


residual = ResidualMLP()
print(residual)
print('total learned numbers:',
      sum(p.numel() for p in residual.parameters()))
print('block output:', tuple(residual.block(X).shape))
print('final output:', tuple(residual(X).shape))

### Shape check

What happens if the block changes the width from 8 to 4?

In [ ]:
bad_block = nn.Sequential(
    nn.Linear(8, 4), nn.ReLU(),
)

try:
    bad_block(X) + X
except RuntimeError as error:
    print('RuntimeError:', str(error).split('.')[0])

Adding tensors requires matching shapes. A residual connection cannot change the width unless we also project `x` to the new width.

## 4. Quick comparison

| model | path | learned numbers |
|---|---|---:|
| `BottleneckMLP` | straight `Sequential` | 217 |
| `ResidualMLP` | block + skip connection | 153 |

Use `Sequential` for a straight stack. Use `nn.Module` when the forward pass branches, merges, or adds a skip connection.

## Exit ticket

1. What are the weight shape, bias shape, and output shape of `nn.Linear(8, 16)`?
2. How many learned numbers does `nn.Linear(8, 16)` have?
3. When do you need a custom `nn.Module` instead of `nn.Sequential`?
4. Why must `block(x)` have the same width as `x` in a residual connection?